In [177]:

from __future__ import annotations

from pathlib import Path
from typing import Iterable, Literal

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib.patches import Rectangle
from scipy.stats import rankdata

import ipywidgets as widgets
from IPython.display import display, clear_output

from HPC_2P_analysis.utils.config import (
    get_itr_index,
    get_one_index,
    get_data_track_type,
    get_data_behavior_type,
    resolve_type_patterns,
    index_to_type,
)

from HPC_2P_analysis.utils.loadData import (
    load_raw_data,
    load_screen_data,
)

from HPC_2P_analysis.utils.processData import (
    align_track_hpc,
)

In [178]:
# =============================================================================
# Data loading
# =============================================================================

def load_browse_data(
    file_path: str | Path,
    *,
    screen_mode: str = "masked",
) -> dict:
    """
    Load data for neuron browsing.

    .mat:
        load raw data.

    .pkl / .pickle:
        load screen result.

    screen_mode:
        "masked" / "screened":
            use masked_attracted_data.

        "raw" / "original":
            use attracted_data.

    This function does not inspect or modify trial index.
    """
    file_path = Path(file_path)
    suffix = file_path.suffix.lower()

    if suffix == ".mat":
        return load_raw_data(file_path)

    if suffix in {".pkl", ".pickle"}:
        return load_screen_data(
            file_path,
            screen_mode=screen_mode,
        )

    raise ValueError(
        f"Unsupported file suffix: {suffix}. "
        "Expected .mat, .pkl, or .pickle."
    )


def _as_tuple(x) -> tuple:
    """
    Normalize input.

    None:
        ("*",)

    "CAB":
        ("CAB",)

    ["CAB", "CBA"]:
        ("CAB", "CBA")
    """
    if x is None:
        return ("*",)

    if isinstance(x, str):
        return (x,)

    return tuple(x)


# =============================================================================
# Track parsing / grouping / visual style
# =============================================================================

# Pair color rule:
#   CAB / CBA -> blue
#   ACB / BCA -> yellow
#   ABC / BAC -> red
#   couple_*  -> gray
_TRACK_PAIR_COLOR = {
    "CAB": "#1f77b4", 
    "ACB": "#ff7f0e", 
    "ABC": "#d62728",
}

_COUPLE_COLOR = "#9e9e9e"

# Behavior linestyle rule:
#   NoReward -> dashed
#   others   -> solid
_BEHAVIOR_LINESTYLE = {
    "Correct": "-",
    "FalseAlarm": "-",
    "Miss": "-",
    "NoReward": "--",
}


def _strip_couple_prefix(track_name: str) -> tuple[str, bool]:
    """
    Remove optional couple prefix.

    Examples
    --------
    CAB:
        ("CAB", False)

    couple_ACB:
        ("ACB", True)

    couple-ACB:
        ("ACB", True)
    """
    track_name = str(track_name)

    if track_name.startswith("couple_"):
        return track_name[len("couple_"):], True

    if track_name.startswith("couple-"):
        return track_name[len("couple-"):], True

    return track_name, False


def _is_couple_track(track_name: str) -> bool:
    """
    Whether this track is a couple track.
    """
    _, is_couple = _strip_couple_prefix(track_name)
    return is_couple


def _is_a_before_b(track_name: str) -> bool:
    """
    Return whether A appears before B in the track name.
    """
    core, _ = _strip_couple_prefix(track_name)

    if "A" not in core or "B" not in core:
        raise ValueError(
            f"Cannot infer AB/BA order from track name {track_name!r}. "
            "Track name must contain both A and B."
        )

    return core.index("A") < core.index("B")


def _base_pair_key(track_name: str) -> str:
    """
    Canonical AB-pair key without couple prefix.

    Examples
    --------
    CAB / CBA:
        CAB

    ACB / BCA:
        ACB

    ABC / BAC:
        ABC

    couple_ACB / couple_BCA:
        ACB
    """
    core, _ = _strip_couple_prefix(track_name)

    if "A" not in core or "B" not in core:
        raise ValueError(
            f"Cannot infer pair key from track name {track_name!r}."
        )

    chars = list(core)

    a_idx = core.index("A")
    b_idx = core.index("B")

    lo = min(a_idx, b_idx)
    hi = max(a_idx, b_idx)

    chars[lo] = "A"
    chars[hi] = "B"

    return "".join(chars)


def _group_pair_key(track_name: str) -> str:
    """
    Pair key used for grouping.

    Important
    ---------
    In 8-track data:
        couple_ACB + couple_BCA
    and:
        ACB + BCA
    are separate groups.
    """
    key = _base_pair_key(track_name)
    _, is_couple = _strip_couple_prefix(track_name)

    if is_couple:
        return f"couple_{key}"

    return key


def _track_color(track_name: str) -> str:
    """
    Color by track identity.

    Rule
    ----
    couple_*:
        gray #9e9e9e

    CAB / CBA:
        blue

    ACB / BCA:
        yellow

    ABC / BAC:
        red
    """
    if _is_couple_track(track_name):
        return _COUPLE_COLOR

    key = _base_pair_key(track_name)

    return _TRACK_PAIR_COLOR.get(
        key,
        "black",
    )


def _pair_color(pair_key: str) -> str:
    """
    Color for merged pair.

    Rule
    ----
    couple_* pair:
        gray #9e9e9e

    CAB / CBA:
        blue

    ACB / BCA:
        yellow

    ABC / BAC:
        red
    """
    pair_key = str(pair_key)

    if pair_key.startswith("couple_") or pair_key.startswith("couple-"):
        return _COUPLE_COLOR

    return _TRACK_PAIR_COLOR.get(
        pair_key,
        "black",
    )


def _behavior_linestyle(
    behavior_name: str | None = None,
    *,
    default: str = "-",
) -> str:
    """
    Linestyle by behavior type.

    NoReward:
        dashed

    Other or unknown behavior:
        solid by default
    """
    if behavior_name is None:
        return default

    behavior_name = str(behavior_name)

    return _BEHAVIOR_LINESTYLE.get(
        behavior_name,
        default,
    )


def _track_linestyle(
    track_name: str | None = None,
    behavior_name: str | None = None,
    *,
    default: str = "-",
) -> str:
    """
    Linestyle for track-level plot.

    Current rule
    ------------
    NoReward:
        dashed

    Other behavior:
        solid

    Notes
    -----
    track_name is kept in the interface for future extension,
    but currently linestyle is determined by behavior_name.
    """
    return _behavior_linestyle(
        behavior_name,
        default=default,
    )


def _pair_linestyle(
    pair_key: str | None = None,
    behavior_name: str | None = None,
    *,
    default: str = "-",
) -> str:
    """
    Linestyle for pair-merged plot.

    Current rule
    ------------
    NoReward:
        dashed

    Other behavior:
        solid

    Notes
    -----
    pair_key is kept in the interface for future extension,
    but currently linestyle is determined by behavior_name.
    """
    return _behavior_linestyle(
        behavior_name,
        default=default,
    )


def _pretty_track_label(track_name: str) -> str:
    """
    Make readable track label.
    """
    return str(track_name).replace("couple_", "couple-")


def _pair_label(track_names: list[str]) -> str:
    """
    Legend label for merged tracks.

    Examples
    --------
    ["CAB", "CBA"]:
        CAB + CBA

    ["couple_ACB", "couple_BCA"]:
        couple-ACB + couple-BCA
    """
    return " + ".join(
        _pretty_track_label(track_name)
        for track_name in track_names
    )


def get_selected_track_names(
    data: dict,
    ana_tt: Iterable[str] = ("*",),
) -> list[str]:
    """
    Resolve selected track names from current data's real track_type.

    track_type is read from:
        data["file_info"]
    """
    ana_tt = _as_tuple(ana_tt)
    track_type = get_data_track_type(data)

    return resolve_type_patterns(
        ana_tt,
        track_type,
    )


def get_ab_ba_track_groups(
    data: dict,
    ana_tt: Iterable[str] = ("*",),
) -> dict[str, list[str]]:
    """
    Split selected tracks into AB and BA groups.

    ana_tt is applied first.
    """
    selected_tracks = get_selected_track_names(
        data,
        ana_tt=ana_tt,
    )

    groups = {
        "AB": [],
        "BA": [],
    }

    for track_name in selected_tracks:
        if _is_a_before_b(track_name):
            groups["AB"].append(track_name)
        else:
            groups["BA"].append(track_name)

    return groups


def get_pair_track_groups(
    data: dict,
    ana_tt: Iterable[str] = ("*",),
) -> dict[str, list[str]]:
    """
    Group selected tracks into AB/BA counterpart pairs.

    ana_tt is applied first.

    Examples
    --------
    6-track file, ana_tt=("*",):
        CAB: ["CAB", "CBA"]
        ACB: ["ACB", "BCA"]
        ABC: ["ABC", "BAC"]

    8-track file, ana_tt=("couple_*",):
        couple_ACB: ["couple_ACB", "couple_BCA"]

    8-track file, ana_tt=("*",):
        couple_ACB: ["couple_ACB", "couple_BCA"]
        CAB: ["CAB", "CBA"]
        ACB: ["ACB", "BCA"]
        ABC: ["ABC", "BAC"]
    """
    selected_tracks = get_selected_track_names(
        data,
        ana_tt=ana_tt,
    )

    pair_groups: dict[str, list[str]] = {}

    for track_name in selected_tracks:
        key = _group_pair_key(track_name)

        if key not in pair_groups:
            pair_groups[key] = []

        pair_groups[key].append(track_name)

    return pair_groups


def _infer_n_neurons_from_firing(
    data: dict,
    *,
    source_key: str,
    ana_tt,
    ana_bt,
) -> int:
    """
    Infer neuron number from a firing-like object array.

    source_key can be:
        "firing"
        "smooth_firing"
        "aligned_firing"
    """
    if source_key not in data:
        raise KeyError(
            f"data does not contain {source_key!r}."
        )

    for tt_idx, bt_idx in get_itr_index(data, ana_tt, ana_bt):
        arr = data[source_key][tt_idx, bt_idx]

        if arr is not None:
            return arr.shape[0]

    raise ValueError(
        f"No valid condition found for source_key={source_key!r}, "
        f"ana_tt={ana_tt}, ana_bt={ana_bt}."
    )


def _format_file_info(data: dict) -> None:
    """
    Print compact file information.
    """
    file_info = data.get("file_info", {})

    if len(file_info) == 0:
        return

    print(f"mouse_id        : {file_info.get('mouse_id', 'NA')}")
    print(f"task_type       : {file_info.get('task_type', 'NA')}")
    print(f"reward_mode     : {file_info.get('reward_mode', 'NA')}")
    print(f"track_type_id   : {file_info.get('track_type_id', 'NA')}")
    print(f"behavior_type_id: {file_info.get('behavior_type_id', 'NA')}")

In [179]:
# =============================================================================
# Common plot helpers
# =============================================================================

def _draw_hpc_zones(
    ax,
    zones,
    *,
    selected_zone_colors: dict[int, str] | None = None,
    h_frac: float = 0.05,
):
    """
    Draw reward zones on one axis.

    This only visualizes data["zones"].
    """
    if zones is None:
        return

    if selected_zone_colors is None:
        selected_zone_colors = {
            0: "#1f77b4",
            2: "#ff7f0e",
            4: "#d62728",
        }

    for zone_idx, color in selected_zone_colors.items():
        if zone_idx >= len(zones):
            continue

        row = zones[zone_idx]

        if row is None:
            continue

        row = np.asarray(row).ravel()

        if len(row) < 2:
            continue

        start_x = float(row[0])
        end_x = float(row[1])

        ax.axvline(
            start_x,
            linestyle="--",
            color="k",
            linewidth=0.8,
        )

        ax.axvline(
            end_x,
            linestyle="--",
            color="k",
            linewidth=0.8,
        )

        rect = Rectangle(
            (start_x, 1.0 - h_frac),
            end_x - start_x,
            h_frac,
            transform=ax.get_xaxis_transform(),
            facecolor=color,
            edgecolor="none",
            clip_on=False,
            zorder=6,
        )

        ax.add_patch(rect)


def _neuron_title(
    data: dict,
    neuron_id: int,
) -> str:
    """
    Title with neuron id and optional cell id.
    """
    if (
        "cell_ids" in data
        and data["cell_ids"] is not None
        and neuron_id < len(data["cell_ids"])
    ):
        return f"Neuron #{neuron_id} (cell id: {data['cell_ids'][neuron_id]})"

    return f"Neuron #{neuron_id}"


def _make_browser_figure(
    n_panels: int,
    *,
    figsize=None,
    single_figsize=(9.2, 4.0),
    double_figsize=(13.2, 4.0),
    sharey: bool = True,
):
    """
    Create figure with size adjusted for one-panel or two-panel plots.

    One-panel:
        wider and a bit shorter.

    Two-panel:
        much wider and a bit shorter for AB / BA.
    """
    if figsize is None:
        if n_panels == 1:
            figsize = single_figsize
        elif n_panels == 2:
            figsize = double_figsize
        else:
            figsize = (6.0 * n_panels, 4.0)

    fig, axes = plt.subplots(
        1,
        n_panels,
        figsize=figsize,
        sharey=sharey if n_panels > 1 else False,
        squeeze=False,
    )

    return fig, axes.ravel()


def _collect_legend_handles_labels(axes):
    """
    Collect deduplicated legend handles and labels.
    """
    if not isinstance(axes, (list, tuple, np.ndarray)):
        axes = [axes]

    handles_all = []
    labels_all = []

    for ax in axes:
        handles, labels = ax.get_legend_handles_labels()

        for handle, label in zip(handles, labels):
            if label not in labels_all:
                handles_all.append(handle)
                labels_all.append(label)

    return handles_all, labels_all


def _add_axis_top_legend(
    ax,
    *,
    fontsize: int = 11,
    ncol: int | None = None,
):
    """
    Add legend above one axis.

    This is used for AB / BA separated plots.
    Each subplot has its own legend.
    """
    handles, labels = ax.get_legend_handles_labels()

    if len(handles) == 0:
        return None

    if ncol is None:
        ncol = min(len(labels), 3)

    legend = ax.legend(
        handles,
        labels,
        loc="lower center",
        bbox_to_anchor=(0.5, 1.16),
        ncol=ncol,
        frameon=False,
        fontsize=fontsize,
        handlelength=1.5,
        columnspacing=1.0,
        handletextpad=0.5,
        borderaxespad=0.0,
    )

    return legend


def _add_shared_legend(
    fig,
    axes,
    *,
    mode: Literal["top", "right", "none"] = "top",
    fontsize: int = 11,
    ncol: int | None = None,
):
    """
    Add shared legend.

    Mainly used for the single-panel merged plot.

    mode="top":
        horizontal legend above the figure.

    mode="right":
        legend close to the right side.

    mode="none":
        no legend.
    """
    if mode == "none":
        return None

    handles, labels = _collect_legend_handles_labels(axes)

    if len(handles) == 0:
        return None

    if ncol is None:
        if mode == "top":
            ncol = min(len(labels), 4)
        else:
            ncol = 1

    if mode == "top":
        return fig.legend(
            handles,
            labels,
            loc="upper center",
            bbox_to_anchor=(0.5, 0.90),
            ncol=ncol,
            frameon=False,
            fontsize=fontsize,
            handlelength=1.5,
            columnspacing=1.3,
            handletextpad=0.5,
        )

    if mode == "right":
        return fig.legend(
            handles,
            labels,
            loc="center left",
            bbox_to_anchor=(0.86, 0.50),
            ncol=ncol,
            frameon=False,
            fontsize=fontsize,
            handlelength=1.5,
            columnspacing=1.0,
            handletextpad=0.5,
        )

    raise ValueError("legend_mode must be 'top', 'right', or 'none'.")


def _apply_browser_axis_style(
    ax,
    *,
    xlabel: str = "Position (cm)",
    ylabel: str | None = None,
    space_unit: float = 2.0,
    axis_label_fontsize: int = 15,
    tick_fontsize: int = 13,
):
    """
    Apply common axis style.
    """
    ax.set_xlabel(
        xlabel,
        fontsize=axis_label_fontsize,
    )

    if ylabel is not None:
        ax.set_ylabel(
            ylabel,
            fontsize=axis_label_fontsize,
        )

    ax.tick_params(
        axis="both",
        labelsize=tick_fontsize,
    )

    ax.xaxis.set_major_formatter(
        mticker.FuncFormatter(
            lambda x, pos: f"{x * space_unit:g}"
        )
    )

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.xaxis.set_ticks_position("bottom")
    ax.yaxis.set_ticks_position("left")


def _finish_browser_layout(
    fig,
    *,
    n_panels: int,
    legend_mode: Literal["top", "right", "none", "panel_top"] = "top",
    has_suptitle: bool = True,
):
    """
    Finish layout.

    panel_top:
        legends are attached to each subplot.
        Used for AB / BA separated plot.

    top:
        shared legend above figure.
        Used for merged single-panel plot.
    """
    if legend_mode == "panel_top":
        fig.subplots_adjust(
            left=0.075,
            right=0.985,
            bottom=0.16,
            top=0.70 if has_suptitle else 0.78,
            wspace=0.24,
        )

    elif legend_mode == "top":
        fig.subplots_adjust(
            left=0.09 if n_panels == 1 else 0.075,
            right=0.985,
            bottom=0.16,
            top=0.74 if has_suptitle else 0.82,
            wspace=0.24,
        )

    elif legend_mode == "right":
        fig.subplots_adjust(
            left=0.09 if n_panels == 1 else 0.075,
            right=0.84,
            bottom=0.16,
            top=0.82 if has_suptitle else 0.90,
            wspace=0.24,
        )

    elif legend_mode == "none":
        fig.subplots_adjust(
            left=0.09 if n_panels == 1 else 0.075,
            right=0.985,
            bottom=0.16,
            top=0.84 if has_suptitle else 0.92,
            wspace=0.24,
        )

    else:
        raise ValueError(
            "legend_mode must be 'top', 'right', 'none', or 'panel_top'."
        )

In [180]:
# =============================================================================
# AB / BA separated plot
# =============================================================================

def plot_hpc_neuron_ab_ba(
    data: dict,
    neuron_id: int,
    ana_tt: Iterable[str] = ("*",),
    ana_bt: Iterable[str] = ("Correct",),
    *,
    show_sem: bool = True,
    show_zones: bool = True,
    space_unit: float = 2.0,
    alpha: float = 0.9,
    lw: float = 2.0,
    figsize=None,
    title_fontsize: int = 14,
    axis_label_fontsize: int = 15,
    tick_fontsize: int = 13,
    suptitle_fontsize: int = 16,
    legend_fontsize: int = 11,
    legend_mode: Literal["panel_top", "none"] = "panel_top",
    show_legend: bool = True,
):
    """
    Plot one HPC neuron in AB / BA panels.

    ana_tt controls which tracks are included before AB/BA grouping.

    Legend
    ------
    For AB / BA separated plot, legends are shown separately above each subplot.
    """
    if "aligned_firing" not in data:
        raise KeyError(
            "data does not contain 'aligned_firing'. "
            "Please run align_track_hpc(data) first."
        )

    if "firing_std" not in data:
        raise KeyError(
            "data does not contain 'firing_std'. "
            "Please run align_track_hpc(data) first."
        )

    ana_tt = _as_tuple(ana_tt)
    ana_bt = _as_tuple(ana_bt)

    groups = get_ab_ba_track_groups(
        data,
        ana_tt=ana_tt,
    )

    fig, axes = _make_browser_figure(
        2,
        figsize=figsize,
        double_figsize=(13.2, 4.0),
        sharey=True,
    )

    effective_legend_mode = legend_mode if show_legend else "none"

    def plot_one_panel(
        ax,
        track_names: list[str],
        panel_title: str,
    ) -> bool:
        found_any = False

        for tt_name in track_names:
            for tt_idx, bt_idx in get_itr_index(data, [tt_name], ana_bt):
                fr_mean = data["aligned_firing"][tt_idx, bt_idx]
                fr_sem = data["firing_std"][tt_idx, bt_idx]

                if fr_mean is None:
                    continue

                if not (0 <= neuron_id < fr_mean.shape[0]):
                    raise ValueError(
                        f"neuron_id {neuron_id} out of range. "
                        f"Valid range: [0, {fr_mean.shape[0] - 1}]"
                    )

                y = fr_mean[neuron_id]

                if fr_sem is None:
                    yerr = np.zeros_like(y)
                else:
                    yerr = fr_sem[neuron_id]

                x = np.arange(y.shape[0])

                color = _track_color(tt_name)
                
                track_name, behavior_name = index_to_type(
                    data,
                    tt_idx,
                    bt_idx,
                )

                ax.plot(
                    x,
                    y,
                    color=color,
                    linestyle=_track_linestyle(tt_name, behavior_name),
                    linewidth=lw,
                    alpha=alpha,
                    label=_pretty_track_label(tt_name),
                )

                if show_sem:
                    ax.fill_between(
                        x,
                        y - yerr,
                        y + yerr,
                        color=color,
                        alpha=0.16,
                    )

                found_any = True

        if show_zones:
            _draw_hpc_zones(
                ax,
                data.get("zones", None),
            )

        ax.set_title(
            panel_title,
            fontsize=title_fontsize,
            pad=8,
        )

        _apply_browser_axis_style(
            ax,
            xlabel="Position (cm)",
            ylabel=None,
            space_unit=space_unit,
            axis_label_fontsize=axis_label_fontsize,
            tick_fontsize=tick_fontsize,
        )

        if effective_legend_mode == "panel_top":
            _add_axis_top_legend(
                ax,
                fontsize=legend_fontsize,
            )

        return found_any

    ok_ab = plot_one_panel(
        axes[0],
        groups["AB"],
        "AB",
    )

    ok_ba = plot_one_panel(
        axes[1],
        groups["BA"],
        "BA",
    )

    if not (ok_ab or ok_ba):
        raise ValueError(
            f"No valid firing array found for neuron={neuron_id}, "
            f"ana_tt={ana_tt}, ana_bt={ana_bt}."
        )

    axes[0].set_ylabel(
        "Deconvolved activity",
        fontsize=axis_label_fontsize,
    )

    fig.suptitle(
        _neuron_title(data, neuron_id),
        fontsize=suptitle_fontsize,
        y=0.985,
    )

    _finish_browser_layout(
        fig,
        n_panels=2,
        legend_mode=effective_legend_mode,
        has_suptitle=True,
    )

    plt.show()

    return fig, axes


def browse_hpc_neurons_ab_ba(
    data: dict,
    ana_tt: Iterable[str] = ("*",),
    ana_bt: Iterable[str] = ("Correct",),
    *,
    gaussian_sigma: float = 2,
    show_sem: bool = True,
    show_zones: bool = True,
    space_unit: float = 2.0,
    alpha: float = 0.9,
    lw: float = 2.0,
    figsize=None,
    id_min: int = 0,
    id_max: int | None = None,
    title_fontsize: int = 14,
    axis_label_fontsize: int = 15,
    tick_fontsize: int = 13,
    suptitle_fontsize: int = 16,
    legend_fontsize: int = 11,
    legend_mode: Literal["panel_top", "none"] = "panel_top",
    show_legend: bool = True,
):
    """
    Browse neurons with AB / BA separated panels.

    ana_tt selects which tracks to plot before AB/BA grouping.
    """
    ana_tt = _as_tuple(ana_tt)
    ana_bt = _as_tuple(ana_bt)

    data = align_track_hpc(
        dict(data),
        gaussian_sigma=gaussian_sigma,
    )

    n_neurons = _infer_n_neurons_from_firing(
        data,
        source_key="aligned_firing",
        ana_tt=ana_tt,
        ana_bt=ana_bt,
    )

    if id_max is None:
        id_max = n_neurons - 1

    if id_min < 0 or id_max >= n_neurons or id_min > id_max:
        raise ValueError(
            f"Invalid id range: [{id_min}, {id_max}], "
            f"valid range is [0, {n_neurons - 1}]"
        )

    track_type = get_data_track_type(data)
    behavior_type = get_data_behavior_type(data)
    selected_tracks = get_selected_track_names(data, ana_tt=ana_tt)
    groups = get_ab_ba_track_groups(data, ana_tt=ana_tt)

    print("=" * 80)
    print("HPC neuron browser | AB / BA separated")
    print(f"n_neurons       : {n_neurons}")
    print(f"track_type      : {track_type}")
    print(f"selected_tracks : {selected_tracks}")
    print(f"behavior_type   : {behavior_type}")
    print(f"AB tracks       : {groups['AB']}")
    print(f"BA tracks       : {groups['BA']}")
    print(f"ana_tt          : {ana_tt}")
    print(f"ana_bt          : {ana_bt}")
    print(f"gaussian_sigma  : {gaussian_sigma}")
    _format_file_info(data)
    print("=" * 80)

    slider = widgets.IntSlider(
        value=id_min,
        min=id_min,
        max=id_max,
        step=1,
        description="Neuron",
        continuous_update=False,
        layout=widgets.Layout(width="500px"),
    )

    btn_prev = widgets.Button(
        description="◀ Prev",
        layout=widgets.Layout(width="90px"),
    )

    btn_next = widgets.Button(
        description="Next ▶",
        layout=widgets.Layout(width="90px"),
    )

    out = widgets.Output()

    def render(_=None):
        with out:
            clear_output(wait=True)

            plot_hpc_neuron_ab_ba(
                data,
                neuron_id=slider.value,
                ana_tt=ana_tt,
                ana_bt=ana_bt,
                show_sem=show_sem,
                show_zones=show_zones,
                space_unit=space_unit,
                alpha=alpha,
                lw=lw,
                figsize=figsize,
                title_fontsize=title_fontsize,
                axis_label_fontsize=axis_label_fontsize,
                tick_fontsize=tick_fontsize,
                suptitle_fontsize=suptitle_fontsize,
                legend_fontsize=legend_fontsize,
                legend_mode=legend_mode,
                show_legend=show_legend,
            )

    def on_prev(_):
        slider.value = max(
            slider.min,
            slider.value - 1,
        )

    def on_next(_):
        slider.value = min(
            slider.max,
            slider.value + 1,
        )

    btn_prev.on_click(on_prev)
    btn_next.on_click(on_next)

    slider.observe(
        render,
        names="value",
    )

    display(
        widgets.HBox(
            [btn_prev, slider, btn_next]
        ),
        out,
    )

    render()


def browse_hpc_neurons_ab_ba_from_file(
    file_path: str | Path,
    ana_tt: Iterable[str] = ("*",),
    ana_bt: Iterable[str] = ("Correct",),
    *,
    screen_mode: str = "masked",
    gaussian_sigma: float = 2,
    show_sem: bool = True,
    show_zones: bool = True,
    space_unit: float = 2.0,
    alpha: float = 0.9,
    lw: float = 2.0,
    figsize=None,
    id_min: int = 0,
    id_max: int | None = None,
    title_fontsize: int = 14,
    axis_label_fontsize: int = 15,
    tick_fontsize: int = 13,
    suptitle_fontsize: int = 16,
    legend_fontsize: int = 11,
    legend_mode: Literal["panel_top", "none"] = "panel_top",
    show_legend: bool = True,
):
    """
    Load raw/screen file and browse AB / BA separated curves.

    For screen pkl:
        screen_mode="masked" means using screened neurons.
    """
    data = load_browse_data(
        file_path,
        screen_mode=screen_mode,
    )

    return browse_hpc_neurons_ab_ba(
        data,
        ana_tt=ana_tt,
        ana_bt=ana_bt,
        gaussian_sigma=gaussian_sigma,
        show_sem=show_sem,
        show_zones=show_zones,
        space_unit=space_unit,
        alpha=alpha,
        lw=lw,
        figsize=figsize,
        id_min=id_min,
        id_max=id_max,
        title_fontsize=title_fontsize,
        axis_label_fontsize=axis_label_fontsize,
        tick_fontsize=tick_fontsize,
        suptitle_fontsize=suptitle_fontsize,
        legend_fontsize=legend_fontsize,
        legend_mode=legend_mode,
        show_legend=show_legend,
    )

In [181]:
GeneralizationCorrMethod = Literal["pearson", "spearman"]

# =============================================================================
# Pair-merged data
# =============================================================================

def summarize_pair_trial_counts(
    data: dict,
    *,
    ana_tt: Iterable[str] = ("*",),
    ana_bt: Iterable[str] = ("Correct",),
    source_key: str = "smooth_firing",
) -> pd.DataFrame:
    """
    Summarize trial number for each selected pair / behavior / track.

    Output columns
    --------------
    pair_key:
        CAB / ACB / ABC / couple_ACB

    behavior_name:
        Correct / NoReward / ...

    track_name:
        CAB / CBA / couple_ACB / ...

    n_trial:
        trial number of this track and behavior.

    n_neuron:
        neuron number.

    n_position:
        position bin number.
    """
    if source_key not in data:
        raise KeyError(
            f"data does not contain {source_key!r}. "
            "Please run align_track_hpc(data) first if using smooth_firing."
        )

    ana_tt = _as_tuple(ana_tt)
    ana_bt = _as_tuple(ana_bt)

    pair_groups = get_pair_track_groups(
        data,
        ana_tt=ana_tt,
    )

    rows = []

    for pair_key, tracks in pair_groups.items():
        for behavior_name in ana_bt:
            for track_name in tracks:
                tt_idx, bt_idx = get_one_index(
                    data,
                    track_name,
                    behavior_name,
                )

                fr = data[source_key][tt_idx, bt_idx]

                if fr is None:
                    rows.append({
                        "pair_key": pair_key,
                        "behavior_name": behavior_name,
                        "track_name": track_name,
                        "n_trial": 0,
                        "n_neuron": np.nan,
                        "n_position": np.nan,
                        "valid": False,
                    })
                    continue

                fr = np.asarray(fr)

                if fr.ndim != 3:
                    raise ValueError(
                        f"Expected {source_key}[{track_name}, {behavior_name}] "
                        f"to be 3D, got shape {fr.shape}."
                    )

                rows.append({
                    "pair_key": pair_key,
                    "behavior_name": behavior_name,
                    "track_name": track_name,
                    "n_trial": int(fr.shape[1]),
                    "n_neuron": int(fr.shape[0]),
                    "n_position": int(fr.shape[2]),
                    "valid": bool(fr.shape[1] > 0),
                })

    return pd.DataFrame(rows)


def _format_trial_count_summary(
    trial_count_df: pd.DataFrame,
) -> str:
    """
    Format trial count table for printing.
    """
    if trial_count_df is None or len(trial_count_df) == 0:
        return "[Trial count] empty"

    lines = []
    lines.append("[Trial count]")

    for (pair_key, behavior_name), sub in trial_count_df.groupby(
        ["pair_key", "behavior_name"],
        sort=False,
    ):
        total = int(sub["n_trial"].sum())

        track_parts = [
            f"{row.track_name}={int(row.n_trial)}"
            for row in sub.itertuples(index=False)
        ]

        lines.append(
            f"  {pair_key} | {behavior_name} | "
            f"total={total} | "
            + ", ".join(track_parts)
        )

    return "\n".join(lines)


def _merge_pair_trials_for_behavior(
    data: dict,
    *,
    pair_tracks: list[str],
    behavior_name: str,
    source_key: str = "smooth_firing",
) -> tuple[np.ndarray | None, dict[str, int]]:
    """
    Merge trials from selected counterpart tracks for one behavior.

    Output
    ------
    merged_fr:
        shape = (n_neuron, n_trial_total, n_position)

    n_trial_by_track:
        {track_name: n_trial}
    """
    blocks = []
    n_trial_by_track = {}

    for track_name in pair_tracks:
        tt_idx, bt_idx = get_one_index(
            data,
            track_name,
            behavior_name,
        )

        fr = data[source_key][tt_idx, bt_idx]

        if fr is None:
            n_trial_by_track[track_name] = 0
            continue

        fr = np.asarray(fr)

        if fr.ndim != 3:
            raise ValueError(
                f"Expected {source_key}[{track_name}, {behavior_name}] "
                f"to be 3D, got shape {fr.shape}."
            )

        n_trial_by_track[track_name] = int(fr.shape[1])

        if fr.shape[1] == 0:
            continue

        blocks.append(fr)

    if len(blocks) == 0:
        return None, n_trial_by_track

    n_neuron_set = {
        block.shape[0]
        for block in blocks
    }

    n_position_set = {
        block.shape[2]
        for block in blocks
    }

    if len(n_neuron_set) != 1 or len(n_position_set) != 1:
        raise ValueError(
            "Cannot merge pair trials because firing shapes are inconsistent. "
            f"Shapes: {[block.shape for block in blocks]}"
        )

    merged_fr = np.concatenate(
        blocks,
        axis=1,
    )

    return merged_fr, n_trial_by_track


def compute_pair_merged_mean_sem(
    data: dict,
    *,
    ana_tt: Iterable[str] = ("*",),
    ana_bt: Iterable[str] = ("Correct",),
    source_key: str = "smooth_firing",
) -> dict[tuple[str, str], dict]:
    """
    Compute pair-merged mean and SEM.

    ana_tt first filters current data's real track_type.

    Mean / SEM rule
    ---------------
    Trials are concatenated first:
        merged = concatenate([fr_AB, fr_BA], axis=1)

    Then:
        mean = mean across merged trials
        sem = std across merged trials / sqrt(n_trial_total)
    """
    if source_key not in data:
        raise KeyError(
            f"data does not contain {source_key!r}. "
            "Please run align_track_hpc(data) first if using smooth_firing."
        )

    ana_tt = _as_tuple(ana_tt)
    ana_bt = _as_tuple(ana_bt)

    pair_groups = get_pair_track_groups(
        data,
        ana_tt=ana_tt,
    )

    out = {}

    for pair_key, tracks in pair_groups.items():
        for behavior_name in ana_bt:
            merged_fr, n_trial_by_track = _merge_pair_trials_for_behavior(
                data,
                pair_tracks=tracks,
                behavior_name=behavior_name,
                source_key=source_key,
            )

            if merged_fr is None:
                continue

            mean = np.nanmean(
                merged_fr,
                axis=1,
            )

            sem = (
                np.nanstd(
                    merged_fr,
                    axis=1,
                    ddof=0,
                )
                / np.sqrt(merged_fr.shape[1])
            )

            out[(pair_key, behavior_name)] = {
                "tracks": list(tracks),
                "label": _pair_label(tracks),
                "mean": mean,
                "sem": sem,
                "n_trial": int(merged_fr.shape[1]),
                "n_trial_by_track": dict(n_trial_by_track),
            }

    return out


# =============================================================================
# Pair-merged plot
# =============================================================================

def plot_hpc_neuron_pair_merged(
    data: dict,
    neuron_id: int,
    ana_tt: Iterable[str] = ("*",),
    ana_bt: Iterable[str] = ("Correct",),
    *,
    source_key: str = "smooth_firing",
    show_sem: bool = True,
    show_zones: bool = True,
    space_unit: float = 2.0,
    alpha: float = 0.9,
    lw: float = 2.2,
    figsize=None,
    title_fontsize: int = 16,
    axis_label_fontsize: int = 15,
    tick_fontsize: int = 13,
    legend_fontsize: int = 11,
    legend_mode: Literal["top", "right", "none"] = "top",
    show_legend: bool = True,
):
    """
    Plot pair-merged activity for one neuron.

    Title:
        only neuron id and optional cell id.

    Legend:
        only track pair label, e.g. CAB + CBA.
        No n_trial in legend.
    """
    ana_tt = _as_tuple(ana_tt)
    ana_bt = _as_tuple(ana_bt)

    pair_stats = compute_pair_merged_mean_sem(
        data,
        ana_tt=ana_tt,
        ana_bt=ana_bt,
        source_key=source_key,
    )

    if len(pair_stats) == 0:
        raise ValueError(
            f"No valid pair-merged firing found for "
            f"ana_tt={ana_tt}, ana_bt={ana_bt}."
        )

    fig, axes = _make_browser_figure(
        1,
        figsize=figsize,
        single_figsize=(9.2, 4.0),
        sharey=False,
    )

    ax = axes[0]
    found_any = False

    for (pair_key, behavior_name), item in pair_stats.items():
        mean = item["mean"]
        sem = item["sem"]

        if not (0 <= neuron_id < mean.shape[0]):
            raise ValueError(
                f"neuron_id {neuron_id} out of range. "
                f"Valid range: [0, {mean.shape[0] - 1}]"
            )

        y = mean[neuron_id]
        yerr = sem[neuron_id]
        x = np.arange(y.shape[0])

        label = item["label"]

        if len(ana_bt) > 1:
            label = f"{label} | {behavior_name}"

        ax.plot(
            x,
            y,
            color=_pair_color(pair_key),
            linestyle=_pair_linestyle(pair_key, behavior_name),
            linewidth=lw,
            alpha=alpha,
            label=label,
        )

        if show_sem:
            ax.fill_between(
                x,
                y - yerr,
                y + yerr,
                color=_pair_color(pair_key),
                alpha=0.16,
            )

        found_any = True

    if not found_any:
        raise ValueError("No valid firing array found for this neuron.")

    if show_zones:
        _draw_hpc_zones(
            ax,
            data.get("zones", None),
        )

    _apply_browser_axis_style(
        ax,
        xlabel="Position (cm)",
        ylabel="Deconvolved activity",
        space_unit=space_unit,
        axis_label_fontsize=axis_label_fontsize,
        tick_fontsize=tick_fontsize,
    )

    fig.suptitle(
        _neuron_title(data, neuron_id),
        fontsize=title_fontsize,
        y=0.985,
    )

    effective_legend_mode = legend_mode if show_legend else "none"

    legend = _add_shared_legend(
        fig,
        ax,
        mode=effective_legend_mode,
        fontsize=legend_fontsize,
    )

    _finish_browser_layout(
        fig,
        n_panels=1,
        legend_mode=effective_legend_mode,
        has_suptitle=True,
    )

    plt.show()

    return fig, ax, legend, pair_stats


def browse_hpc_neurons_pair_merged(
    data: dict,
    ana_tt: Iterable[str] = ("*",),
    ana_bt: Iterable[str] = ("Correct",),
    *,
    gaussian_sigma: float = 2,
    show_sem: bool = True,
    show_zones: bool = True,
    space_unit: float = 2.0,
    alpha: float = 0.9,
    lw: float = 2.2,
    figsize=None,
    id_min: int = 0,
    id_max: int | None = None,
    title_fontsize: int = 16,
    axis_label_fontsize: int = 15,
    tick_fontsize: int = 13,
    legend_fontsize: int = 11,
    legend_mode: Literal["top", "right", "none"] = "top",
    show_legend: bool = True,
    print_trial_count: bool = True,
):
    """
    Browse neurons after merging selected AB/BA counterpart trials.

    ana_tt selects which tracks are allowed before pair merging.

    print_trial_count:
        print trial count for each pair / behavior / track.
    """
    ana_tt = _as_tuple(ana_tt)
    ana_bt = _as_tuple(ana_bt)

    data = align_track_hpc(
        dict(data),
        gaussian_sigma=gaussian_sigma,
    )

    n_neurons = _infer_n_neurons_from_firing(
        data,
        source_key="smooth_firing",
        ana_tt=ana_tt,
        ana_bt=ana_bt,
    )

    if id_max is None:
        id_max = n_neurons - 1

    if id_min < 0 or id_max >= n_neurons or id_min > id_max:
        raise ValueError(
            f"Invalid id range: [{id_min}, {id_max}], "
            f"valid range is [0, {n_neurons - 1}]"
        )

    track_type = get_data_track_type(data)
    behavior_type = get_data_behavior_type(data)
    selected_tracks = get_selected_track_names(data, ana_tt=ana_tt)
    pair_groups = get_pair_track_groups(data, ana_tt=ana_tt)

    trial_count_df = summarize_pair_trial_counts(
        data,
        ana_tt=ana_tt,
        ana_bt=ana_bt,
        source_key="smooth_firing",
    )

    print("=" * 80)
    print("HPC neuron browser | AB / BA merged")
    print(f"n_neurons       : {n_neurons}")
    print(f"track_type      : {track_type}")
    print(f"selected_tracks : {selected_tracks}")
    print(f"behavior_type   : {behavior_type}")
    print(f"pair_groups     : {pair_groups}")
    print(f"ana_tt          : {ana_tt}")
    print(f"ana_bt          : {ana_bt}")
    print(f"gaussian_sigma  : {gaussian_sigma}")
    _format_file_info(data)

    if print_trial_count:
        print(_format_trial_count_summary(trial_count_df))

    print("=" * 80)

    slider = widgets.IntSlider(
        value=id_min,
        min=id_min,
        max=id_max,
        step=1,
        description="Neuron",
        continuous_update=False,
        layout=widgets.Layout(width="500px"),
    )

    btn_prev = widgets.Button(
        description="◀ Prev",
        layout=widgets.Layout(width="90px"),
    )

    btn_next = widgets.Button(
        description="Next ▶",
        layout=widgets.Layout(width="90px"),
    )

    out = widgets.Output()

    def render(_=None):
        with out:
            clear_output(wait=True)

            plot_hpc_neuron_pair_merged(
                data,
                neuron_id=slider.value,
                ana_tt=ana_tt,
                ana_bt=ana_bt,
                source_key="smooth_firing",
                show_sem=show_sem,
                show_zones=show_zones,
                space_unit=space_unit,
                alpha=alpha,
                lw=lw,
                figsize=figsize,
                title_fontsize=title_fontsize,
                axis_label_fontsize=axis_label_fontsize,
                tick_fontsize=tick_fontsize,
                legend_fontsize=legend_fontsize,
                legend_mode=legend_mode,
                show_legend=show_legend,
            )

    def on_prev(_):
        slider.value = max(
            slider.min,
            slider.value - 1,
        )

    def on_next(_):
        slider.value = min(
            slider.max,
            slider.value + 1,
        )

    btn_prev.on_click(on_prev)
    btn_next.on_click(on_next)

    slider.observe(
        render,
        names="value",
    )

    display(
        widgets.HBox(
            [btn_prev, slider, btn_next]
        ),
        out,
    )

    render()


def browse_hpc_neurons_pair_merged_from_file(
    file_path: str | Path,
    ana_tt: Iterable[str] = ("*",),
    ana_bt: Iterable[str] = ("Correct",),
    *,
    screen_mode: str = "masked",
    gaussian_sigma: float = 2,
    show_sem: bool = True,
    show_zones: bool = True,
    space_unit: float = 2.0,
    alpha: float = 0.9,
    lw: float = 2.2,
    figsize=None,
    id_min: int = 0,
    id_max: int | None = None,
    title_fontsize: int = 16,
    axis_label_fontsize: int = 15,
    tick_fontsize: int = 13,
    legend_fontsize: int = 11,
    legend_mode: Literal["top", "right", "none"] = "top",
    show_legend: bool = True,
    print_trial_count: bool = True,
):
    """
    Load raw/screen file and browse AB / BA merged pair curves.

    For screen pkl:
        screen_mode="masked" means using screened neurons.
    """
    data = load_browse_data(
        file_path,
        screen_mode=screen_mode,
    )

    return browse_hpc_neurons_pair_merged(
        data,
        ana_tt=ana_tt,
        ana_bt=ana_bt,
        gaussian_sigma=gaussian_sigma,
        show_sem=show_sem,
        show_zones=show_zones,
        space_unit=space_unit,
        alpha=alpha,
        lw=lw,
        figsize=figsize,
        id_min=id_min,
        id_max=id_max,
        title_fontsize=title_fontsize,
        axis_label_fontsize=axis_label_fontsize,
        tick_fontsize=tick_fontsize,
        legend_fontsize=legend_fontsize,
        legend_mode=legend_mode,
        show_legend=show_legend,
        print_trial_count=print_trial_count,
    )

In [182]:
file_path = "../../../data/HPC_2p/screen/HP31/HP31_2025-09-10_75_position_screen.pkl"

In [183]:
# browse_hpc_neurons_pair_merged_from_file(
#     file_path=file_path,
#     ana_tt=('*'),
#     ana_bt=("Correct","NoReward"),
#     screen_mode="masked",
#     gaussian_sigma=2,
# )

In [184]:
# browse_hpc_neurons_ab_ba_from_file(
#     file_path=file_path,
#     ana_tt=("*",),
#     ana_bt=("NoReward",),
#     screen_mode="masked",
#     gaussian_sigma=2,
# )

In [185]:
# browse_hpc_neurons_pair_merged_from_file(
#     file_path=file_path,
#     ana_tt=("*",),
#     ana_bt=("Correct",),
#     screen_mode="masked",
#     gaussian_sigma=2,
# )